# TFM — v2: las piezas del temario que faltaban

Copia de `modelo_rnn_deeplearning.ipynb` con **una sección nueva, la 8f**, y nada más
cambiado. El original se deja intacto porque está en ejecución; cuando esta versión
demuestre que aporta, se funden.

## Qué añade la 8f

Repasando los notebooks de clase del módulo 14 y de deep learning, hay cuatro técnicas que
allí se usan y aquí no se aplicaban:

| técnica | usos en el temario | estado |
|---|---|---|
| `kernel_regularizer=l2` | **16** — la 2ª más usada tras Dropout | no se usaba |
| `return_state` → `initial_state` | **9** | no se usaba |
| `GaussianNoise` | 3 | no se usaba |
| `Attention` | 1 | no se usaba |

Y una de ellas corrige una diferencia arquitectónica real: **el Seq2Seq de la sección 7 no
es un Seq2Seq canónico.** Comprime el encoder a un vector y lo repite con `RepeatVector`;
el de clase pasa los estados `(h, c)` al decoder como `initial_state`, de modo que el
decoder arranca con la memoria del encoder cargada en lugar de en cero.

Se prueban **acumulándose una a una**, así que la diferencia entre dos filas consecutivas
de la tabla es lo que aporta esa pieza concreta. Con 2.388 ejemplos de entrenamiento no
está garantizado que todas ayuden — y que alguna estorbe es tan publicable como lo
contrario.

## Cómo se ejecuta

Igual que el original: elige `MATRIZ` en la celda de configuración y ejecútalo entero. La
8f va después de la 8e, así que necesita `yr` e `inv_r`, que se definen en la 8c.


# TFM — Predicción del precio diario (D+1) con redes neuronales

Universidad Complutense de Madrid · Máster en Big Data y Data Science

**Problema.** A las 12:00 del día D, cuando cierra el mercado diario, hay que emitir de una vez
las **24 horas** de precio del día D+1. No es predicción a un paso: es una salida multi-horizonte
directa, y esa diferencia condiciona toda la arquitectura.

## Relación con los notebooks de clase

| Notebook de clase | Qué se reutiliza | Qué hay que cambiar |
|---|---|---|
| `Introduction_to_RNN_Time_Series` | Ventanas deslizantes, pipeline de series | Allí es *one-step-ahead* recursivo; aquí, salida directa de 24 valores |
| `IMBD_RNN` | LSTM apiladas, Deep RNN | Allí clasificación con `softmax`; aquí regresión con salida lineal |
| `Seq2seq` / `03_deep_learning_text_translation` | Encoder → `RepeatVector` → decoder → `TimeDistributed` | El decoder recibe **exógenas futuras concatenadas**, no solo el contexto repetido |
| `04-attention` | Base teórica para el capítulo de interpretabilidad | — |

## Tres cosas que los notebooks de clase hacen y aquí serían errores

1. **`Bidirectional` en el decoder.** En texto es gratis. En una serie temporal, una capa
   bidireccional sobre la ventana de salida mira horas futuras para predecir la actual. En el
   **encoder** sí vale: ese pasado ya ocurrió entero.
2. **`softmax` + `sparse_categorical_crossentropy`.** Esto es regresión: salida lineal y pérdida
   **Huber**. Con MSE, los picos de 2022 (>500 €/MWh) dominarían el gradiente y el modelo se
   dedicaría a esos días a costa de los 2.000 normales.
3. **Escalar antes de partir.** El notebook de Jena calcula media y desviación sobre el conjunto
   completo. Eso mete información de test en el preprocesado. Aquí el escalador **se ajusta solo
   con train**.

In [ ]:
# ── Configuracion de GPU (WSL2 + RTX 5050) ────────────────────────────────────────────────
import tensorflow as tf

# Sin memory_growth, TF reserva TODA la VRAM al arrancar. Con Chrome, Excel y VS Code
# compitiendo por la misma tarjeta de 8 GB, es la via rapida a un OOM a mitad de entrenamiento.
for g in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(g, True)

gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)
for g in gpus:
    print(" ", tf.config.experimental.get_device_details(g))
if not gpus:
    print("\nAVISO: sin GPU. Comprobaciones:")
    print("  - el kernel debe ser /home/torgi/tf/bin/python (WSL), no un Python de Windows")
    print("  - LD_LIBRARY_PATH debe incluir /usr/lib/wsl/lib y las libs de pip nvidia-*")

# La RTX 5050 es Blackwell (compute capability 12.0) y TF 2.21 no trae kernels compilados para
# esa arquitectura: los genera por JIT desde PTX. La PRIMERA epoca de cada arquitectura nueva
# va lenta; despues se cachea en CUDA_CACHE_PATH y vuela. No es un fallo, es lo esperado.

In [ ]:
import warnings, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore")

SEMILLA = 42
np.random.seed(SEMILLA)
tf.random.set_seed(SEMILLA)

VENTANA_DIAS = 7          # encoder: 7 dias = 168 h, cubre el ciclo semanal completo
EPOCHS = 80
BATCH = 64        # con GPU no hay motivo para 32; menos overhead por batch

# MATRIZ. Las cuatro salen del notebook de depuracion (notebooks/04) y estan pensadas para
# compararse entre si: las tres primeras cambian las columnas con la ventana fija, y la
# cuarta cambia la ventana con las columnas fijas. Ejecuta el notebook entero con cada una
# y compara EN VALIDACION.
#
#   completa   2020 ->   141 inputs   referencia
#   nucleo     2020 ->   112 inputs   sin redundantes ni derivadas: ¿sobran variables?
#   minima     2020 ->    25 inputs   ¿bastan las 25 mejores?
#   moderna    2023 ->   112 inputs   mismo pool que nucleo: ¿estorba la crisis del gas?
#
# `moderna` contra `nucleo` es la comparacion que hay que mirar primero: con todo lo demas
# identico, mide si los 26.000 registros de 2020-2022 -- con 2022 a 2,66 veces el precio de
# 2024 y el tope al gas activo -- ayudan o estorban.
MATRIZ = "nucleo"          # completa | nucleo | minima | moderna

# Encoder multicanal: precio + las series horarias como canales. False vuelve al encoder de
# un solo canal (solo precio) -- es la ablacion que cuantifica cuanto aporta.
ENCODER_MULTICANAL = True

# APAGON IBERICO. Ya NO se excluye: las matrices vienen con la ventana del 28-abr al 1-may
# sustituida en bloque por la semana anterior, y la resaca hasta el 7-may rellenada solo en
# los huecos (ver `scripts/apagon.py`). Asi 2025 queda completo, que es lo que pide el
# protocolo de cierre.
#
# Lo que si sigue abierto es que del 2 al 7 de mayo las columnas con desfase apuntan a dias
# del apagon. Ese dato es REAL -- el sistema de verdad tuvo un cero seis dias antes -- y por
# eso no se toco, pero las filas van marcadas con `ventana_pisa_apagon`. Poner esto a True
# las descarta, y descarta ademas las ventanas del encoder que las pisen.
EXCLUIR_VENTANA_APAGON = False

print("TensorFlow", tf.__version__, "| GPU:", bool(tf.config.list_physical_devices("GPU")))
print("pandas", pd.__version__, "| numpy", np.__version__)
print(f"matriz: {MATRIZ} | encoder multicanal: {ENCODER_MULTICANAL}")

## 1. Carga de la matriz depurada

Las cuatro matrices salen de `notebooks/04_depuracion_matriz_completa.ipynb`, que las construye desde Postgres, imputa y poda. Aquí sólo se leen: si algo falta o tiene nulos, se arregla allí y no aquí.

In [ ]:
# Las matrices salen de `notebooks/04_depuracion_matriz_completa.ipynb`. Si falta alguna,
# ese notebook las regenera enteras desde Postgres.
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "gold").is_dir())
RUTA = REPO / "data" / "gold" / f"matriz_{MATRIZ}.parquet"
assert RUTA.exists(), (f"No encuentro {RUTA}. Ejecuta antes "
                       f"notebooks/04_depuracion_matriz_completa.ipynb")

df = pd.read_parquet(RUTA)
df = df.sort_values(["fecha_objetivo", "hora"]).reset_index(drop=True)

META = json.loads(RUTA.with_suffix(".meta.json").read_text(encoding="utf-8"))

print(f"{RUTA}")
print(f"{df.shape[0]:,} filas x {df.shape[1]} columnas · {META['n_inputs']} inputs")
print(f"rango: {df.fecha_objetivo.min().date()} -> {df.fecha_objetivo.max().date()}")
print(f"aisla: {META['aisla']}")
print(f"hash : {META.get('hash','?')}   (generada {META.get('generada','?')})")
print(f"nulos: {int(df.isna().sum().sum())}")
print()
print(df.split.value_counts().to_string())

# La matriz sale depurada sin un solo nulo. Si aparece alguno es que se ha tocado por el
# camino, y conviene enterarse aqui y no dentro del escalador.
assert df.isna().sum().sum() == 0, "la matriz deberia venir sin nulos"

In [ ]:
# Reparto por la FRONTERA DE FUGA (12:00 del dia D), no por el tipo de dato.
#
# CONVENCION DE DESFASES, medida contra la fuente y no deducida del nombre:
#
#     sufijo    describe el dia      ejemplo verificado
#     _D        T-1  (el dia D)      es_esios_D coincide con el precio de T-1 al 99,9 %
#     _Dm1      T-2                  99,7 %
#     _Dm2      T-3
#     _Dm6      T-7                  99,4 %
#
# El bloque `_D` es el mas fresco que existe, y es HORARIO. En el dataset anterior `pdbc_`
# venia agregado a media diaria y por eso vivia entre los estaticos; ahora tiene sus 24
# valores por dia (mediana medida: 24), asi que meterlo ahi con `.first()` tiraria 23 de
# cada 24 horas. Va al encoder como canales.
CLAVES = ["fecha_pred", "fecha_objetivo", "hora", "target_price", "split"]
BANDERAS = ["imputado_apagon", "ventana_pisa_apagon"]   # trazabilidad, no features

# (a) DECODER -- publicado por adelantado sobre D+1, o del dia D y ya conocido.
#     Precios europeos SIEMPRE del dia D: todas las zonas SDAC se casan a la vez a las
#     12:00, asi que el precio frances de D+1 no existe cuando hay que predecir el español
#     de D+1 -- son el resultado de la MISMA subasta. Los spreads señalan las horas de
#     desacople, que son las de congestion y precio extremo.
#     `*_meteo` es el canal meteorologico empalmado: prevision ECMWF para D+1 desde abril de
#     2024, y ERA5 desfasado antes. `meteo_es_forecast` dice cual de las dos es, y entra
#     como feature porque sin ella el modelo no puede distinguir los dos regimenes.
cols_dec = ([c for c in df.columns if c.endswith(("_prev_mw", "_prev"))]
            + [c for c in df.columns if c.startswith("ree_ntc_")]
            + [c for c in df.columns if c == "es_esios_D"]
            + [c for c in df.columns if c.endswith(("_entsoe_D", "_omie_D"))]
            + [c for c in df.columns if c.startswith("spread_es_")]
            + [c for c in df.columns if c.endswith("_meteo")]
            + [c for c in df.columns if c in ("meteo_es_forecast", "hora_sin", "hora_cos")])
cols_dec = list(dict.fromkeys(cols_dec))

# (b) ENCODER, canal del dia D -- el programa de casacion, hora a hora. El PBF del dia X se
#     publica a las 13:45 de X-1, o sea que el del propio dia D ya esta disponible a las
#     12:00 de D: entra sin fuga. Es informacion que el diseño anterior no tenia.
cols_prog = [c for c in df.columns
             if c.startswith(("pdbc_", "pbfli_", "bil_")) and c.endswith("_D")]

# (c) ENCODER, canal del dia D-1 -- series reales horarias.
cols_dm1 = [c for c in df.columns if c.endswith("_Dm1") and c != "es_esios_Dm1"]

# (d) ESTATICOS -- lo que de verdad es constante dentro del dia. Se comprueba en lugar de
#     suponerse: `capinst_` y `capdisp_` salen diarias, y los testigos de publicacion
#     tambien. Las horarias que caerian aqui se agregan a MEDIA del dia, no a `.first()`,
#     que se quedaria con la hora 0 y para la temperatura eso es la madrugada.
cols_dm2 = [c for c in df.columns if c.endswith("_Dm2")]
candidatas_est = [c for c in df.columns
                  if (c.startswith(("capinst_", "capdisp_", "d1_"))
                      or c in ("gas_mibgas", "co2_eua_dec", "gas_ttf_m1")
                      or c.startswith(("pbf_publicado", "pbf_completo")))] + cols_dm2

if ENCODER_MULTICANAL:
    # Las series horarias viven en el ENCODER como canales. Dejarlas ademas agregadas aqui
    # seria escribir el mismo dato dos veces y con peor resolucion: la LSTM comparte pesos
    # entre los 168 pasos, asi que aprende UNA VEZ "como se comporta la eolica" y lo aplica
    # a todas las horas. En columnas planas, wind_D-1 y wind_D-7 son dos variables sueltas.
    candidatas_est = [c for c in candidatas_est if not c.endswith(("_Dm1", "_Dm6"))]
else:
    candidatas_est += [c for c in df.columns if c.endswith(("_Dm1", "_Dm6"))]
    candidatas_est += cols_prog

candidatas_est = [c for c in dict.fromkeys(candidatas_est)
                  if c not in cols_dec and c not in CLAVES + BANDERAS]

# horaria o diaria: medido, no supuesto
nun = df.groupby("fecha_objetivo")[candidatas_est].nunique().mean()
cols_est = [c for c in candidatas_est if nun[c] <= 1.5]
cols_est_media = [c for c in candidatas_est if nun[c] > 1.5]

const = [c for c in cols_dec + cols_est + cols_est_media if df[c].nunique(dropna=True) <= 1]
cols_dec = [c for c in cols_dec if c not in const]
cols_est = [c for c in cols_est if c not in const]
cols_est_media = [c for c in cols_est_media if c not in const]

usadas = set(cols_dec + cols_prog + cols_dm1 + cols_est + cols_est_media + CLAVES + BANDERAS
             + const)
sobrantes = [c for c in df.select_dtypes("number").columns if c not in usadas]

# `_Dm6` y los precios desfasados NO son un olvido: la ventana del encoder cubre de D-7 a D,
# asi que el dato de hace seis dias ya viaja dentro de ella. Meterlos ademas como columnas
# planas seria escribir lo mismo dos veces, y con peor resolucion.
redundantes = [c for c in sobrantes
               if c.endswith("_Dm6") or c.startswith("es_esios_Dm")]
huerfanas = [c for c in sobrantes if c not in redundantes]

n_eu = sum(1 for c in cols_dec if c.endswith(("_entsoe_D", "_omie_D"))
           or c.startswith("spread_es_"))
n_met = sum(1 for c in cols_dec if c.endswith("_meteo"))
print(f"descartadas por constantes ({len(const)}): {const}")
print()
print(f"decoder   {len(cols_dec):3d}  ({n_eu} precios EU, {n_met} meteo D+1)")
print(f"encoder   {1 + len(cols_prog) + len(cols_dm1):3d} canales  "
      f"(precio + {len(cols_prog)} programa dia D + {len(cols_dm1)} reales dia D-1)")
print(f"estaticos {len(cols_est):3d}  + {len(cols_est_media)} horarias agregadas a media diaria")
print(f"fuera por redundar con la ventana del encoder: {len(redundantes)} "
      f"(`_Dm6` y precios desfasados)")
if huerfanas:
    print()
    print(f"AVISO: {len(huerfanas)} columnas SIN CLASIFICAR, no entran al modelo.")
    print("Si son nuevas, hay que decidir a que bloque van en lugar de dejarlas caer:")
    print(f"   {huerfanas}")

## 2. De tabla larga a tensores — ventanas deslizantes

Es el patrón *Sliding Windows* del notebook de clase, adaptado a salida directa:

- **`X_enc` (n, 168, C)** — los 7 días previos hora a hora. Canal 0 el precio (ventana que
  termina en el día **D**, cuyo precio se casó ayer a las 12:00 y lleva 23 h publicado), y el
  resto las series reales de D−7 a D−1. El día D **no** entra en los canales reales: a las
  12:00 solo han ocurrido sus horas 00–11.
- **`X_dec` (n, 24, f)** — previsiones de D+1 hora a hora + precio de D alineado por hora.
- **`X_est` (n, j)** — lo que no varía dentro del día.
- **`y` (n, 24)** — el target.

**Por qué canales y no columnas con lag.** Con 17 tecnologías × 3 estadísticos × 7 días serían
357 columnas, y además `wind_D-1` y `wind_D-7` serían para el modelo dos variables sin relación
alguna. La LSTM **comparte los mismos pesos en los 168 pasos**: aprende una vez cómo se comporta
la eólica y lo aplica a todas las horas. Pasar de 1 a 21 canales apenas mueve el número de
parámetros.

**Contigüidad.** Al excluir el apagón se abre un hueco en la serie, y `PRECIO[t-V:t]` opera
sobre los días *disponibles*. Sin comprobarlo, la fila posterior al hueco tomaría siete jornadas
no consecutivas: entrenaría igual, sin avisar, y solo se notaría en un rendimiento algo peor.

In [ ]:
dias = np.array(sorted(df.fecha_objetivo.unique()))
n_dias = len(dias)

# LA HORA QUE NO EXISTIO. El domingo del cambio de hora de primavera tiene 23 horas y la
# depuracion elimina esa fila, porque imputarla seria fabricar una hora que el reloj se
# salto. Pero un tensor (dias, 24, canales) es rectangular por definicion: necesita las 24
# casillas aunque una de ellas no corresponda a ningun instante real.
#
# Asi que aqui se rellena, y conviene tener claro que NO es una imputacion de dato: es el
# relleno que exige la representacion. Son 7 casillas por canal en seis anos y medio, y se
# toman de las horas vecinas del mismo dia. La alternativa -- dejar el NaN -- se llevaba por
# delante 57 dias enteros de entrenamiento, porque cada dia incompleto invalida las ocho
# ventanas que lo pisan.
HORAS = list(range(24))

def _rellena_hora_inexistente(p):
    "Interpola a lo largo de las 24 horas de cada dia. p: (dias x 24)."
    return p.reindex(columns=HORAS).interpolate(axis=1, limit_direction="both")

def panel(cols, desfase=0):
    """(fecha_objetivo, hora) -> (n_dias, 24, len(cols)), opcionalmente reindexado.

    `desfase` es cuantos dias hay que restar a `fecha_objetivo` para que la fila quede
    fechada en el dia que DESCRIBE. Con eso, cualquier bloque se puede recortar despues con
    la misma aritmetica que el precio, sin arrastrar el desfase por el resto del notebook.
    """
    h = df[["fecha_objetivo", "hora"] + cols]
    if desfase:
        h = h.assign(fecha_objetivo=h["fecha_objetivo"] - pd.Timedelta(days=desfase))
    p = (h.pivot_table(index="fecha_objetivo", columns="hora", values=cols, aggfunc="mean")
          .reindex(dias))
    return np.stack([_rellena_hora_inexistente(p[c]).to_numpy(dtype="float32")
                     for c in cols], axis=-1)

PRECIO = _rellena_hora_inexistente(
    df.pivot_table(index="fecha_objetivo", columns="hora", values="target_price",
                   aggfunc="mean").reindex(dias)).to_numpy(dtype="float32")
n_dst = int(np.isnan(df.pivot_table(index="fecha_objetivo", columns="hora",
                                    values="target_price", aggfunc="mean")
                       .reindex(dias).reindex(columns=HORAS)).sum().sum())
print(f"casillas rellenadas por el cambio de hora: {n_dst} de {n_dias * 24:,} "
      f"({n_dst / (n_dias * 24) * 100:.3f}%)")
DEC = panel(cols_dec)
EST = (df.groupby("fecha_objetivo")[cols_est].first().reindex(dias).to_numpy(dtype="float32")
       if cols_est else np.zeros((n_dias, 0), dtype="float32"))
if cols_est_media:
    EST = np.concatenate([EST, df.groupby("fecha_objetivo")[cols_est_media].mean()
                          .reindex(dias).to_numpy(dtype="float32")], axis=1)
cols_est = cols_est + cols_est_media

# ── canales del encoder ───────────────────────────────────────────────────────
# Cada bloque se reindexa a la fecha que describe y luego se recorta igual que el precio:
# `[t-V:t]` son los V dias que terminan en D. Sin el reindexado habria que llevar un
# desfase distinto por bloque hasta el final, que es como se cuelan los errores de un dia.
canales = ["precio"]
BLOQUES = []
if ENCODER_MULTICANAL:
    if cols_prog:
        BLOQUES.append(panel(cols_prog, desfase=1))     # `_D` describe T-1
        canales += [c[:-2] + "@D" for c in cols_prog]
    if cols_dm1:
        BLOQUES.append(panel(cols_dm1, desfase=2))      # `_Dm1` describe T-2
        canales += [c[:-4] for c in cols_dm1]

V = VENTANA_DIAS
t_idx = np.arange(V + 1, n_dias)
dias_dt = pd.to_datetime(dias)

# COLA SIN DATO. Reindexar un bloque con desfase de k dias deja los ULTIMOS k dias vacios:
# la fila que describia el 30-jul pasa a estar fechada el 29, y el 30 se queda sin nadie que
# lo describa. Con la ventana llegando hasta el ultimo dia, esos NaN entran al encoder.
#
# El notebook anterior tenia esto mismo con el bloque `_Dm1` (desfase 2) y sin comprobarlo,
# asi que colaba NaN en los dos ultimos dias. Aqui se marcan los dias incompletos de cada
# bloque y se descartan las ventanas que los toquen, en vez de confiar en la aritmetica.
sin_dato = np.isnan(PRECIO).any(axis=1)
for b in BLOQUES:
    sin_dato |= np.isnan(b).any(axis=(1, 2))

excl = np.zeros(n_dias, dtype=bool)
if EXCLUIR_VENTANA_APAGON and "ventana_pisa_apagon" in df.columns:
    marcados = set(pd.to_datetime(
        df.loc[df.ventana_pisa_apagon == 1, "fecha_objetivo"].unique()))
    excl |= np.array([d in marcados for d in dias_dt])

motivos = {"cola sin dato": 0, "ventana no contigua": 0, "apagon": 0}
ok_v = np.ones(len(t_idx), dtype=bool)
for j, t in enumerate(t_idx):
    if sin_dato[t - V:t].any():
        motivos["cola sin dato"] += 1; ok_v[j] = False
    elif (dias_dt[t - 1] - dias_dt[t - V - 1]).days != V:
        motivos["ventana no contigua"] += 1; ok_v[j] = False
    elif excl[t - V:t + 1].any():
        motivos["apagon"] += 1; ok_v[j] = False
print(f"{int((~ok_v).sum())} dias descartados: "
      + ", ".join(f"{v} por {k}" for k, v in motivos.items() if v))
t_idx = t_idx[ok_v]

X_enc = np.stack([
    np.concatenate([PRECIO[t - V:t].reshape(24 * V, 1)]
                   + [b[t - V:t].reshape(24 * V, b.shape[-1]) for b in BLOQUES], axis=-1)
    for t in t_idx])
X_dec, X_est, y = DEC[t_idx], EST[t_idx], PRECIO[t_idx]
fechas = dias[t_idx]

assert not np.isnan(X_enc).any(), "NaN en el encoder: revisa el desfase de algun bloque"
print(f"X_enc {X_enc.shape} | X_dec {X_dec.shape} | X_est {X_est.shape} | y {y.shape}")
print(f"canales del encoder: {len(canales)}")
print(f"primer dia predicho: {pd.Timestamp(fechas[0]).date()}")

In [ ]:
# Un solo NaN en X hace que la perdida salga `nan` desde la primera epoca, SIN lanzar ningun
# error. Es el fallo mas dificil de diagnosticar en Keras: se comprueba antes de entrenar.
for n, a in [("X_enc", X_enc), ("X_dec", X_dec), ("X_est", X_est), ("y", y)]:
    k = int(np.isnan(a).sum())
    print(f"{n:<6} NaN: {k:>7,}  ({100*k/a.size:.3f}%)")

X_dec = np.nan_to_num(X_dec)
X_est = np.nan_to_num(X_est)
ok = ~np.isnan(y).any(axis=1) & ~np.isnan(X_enc).any(axis=(1, 2))
X_enc, X_dec, X_est, y, fechas = X_enc[ok], X_dec[ok], X_est[ok], y[ok], fechas[ok]
print(f"\nfilas utiles: {ok.sum()} de {len(ok)}")

## 3. Split cronológico, purga de columnas y escalado

**El split se corta por día, jamás por fila.** Las 24 horas de un día comparten conjunto de
información; un `train_test_split(shuffle=True)` pondría horas del mismo día a ambos lados y el
modelo "acertaría" copiando de sus vecinas.

**Purga de columnas sin varianza EN TRAIN.** El filtro de constantes de la §1 mira todo el
rango, y eso deja pasar un caso letal: una columna puede ser constante *solo dentro de train* y
variar después. Los indicadores de batería (ESIOS 2166/2167) publican desde el 20-nov-2024 y
train acaba el 31-dic-2024, así que en train valen 0 y en test valen cientos de MW. Al
estandarizar, su desviación de train es 0, se divide entre el epsilon y **todo val/test se
multiplica por un millón**. En la primera ejecución esto produjo `val_loss` de 268.000 y un MAE
de 832.039 €/MWh con la pérdida de entrenamiento en 0,02 — el patrón que delata que el problema
está en el preprocesado y no en el modelo.

**El escalador se ajusta solo con train.** Es la fuga más silenciosa que existe: usar la media
del conjunto completo mete información de test en el preprocesado.

In [ ]:
f = pd.to_datetime(fechas)
tr = f <= "2024-12-31"
va = (f > "2024-12-31") & (f <= "2025-12-31")
te = f > "2025-12-31"
print(f"train {tr.sum()} | val {va.sum()} | test {te.sum()} dias\n")

def purgar(X, nombres, etiqueta):
    "Descarta columnas cuya desviacion en TRAIN es cero (ver nota de la celda anterior)."
    ejes = tuple(range(X.ndim - 1))
    sd = X[tr].std(axis=ejes)
    malas = set(np.where(sd < 1e-8)[0].tolist())
    if malas:
        print(f"[{etiqueta}] {len(malas)} columnas sin varianza en train, descartadas:")
        print("   " + ", ".join(nombres[i] for i in sorted(malas)))
    buenas = [i for i in range(X.shape[-1]) if i not in malas]
    return X[..., buenas], [nombres[i] for i in buenas]

X_dec, cols_dec = purgar(X_dec, cols_dec, "decoder")
X_est, cols_est = purgar(X_est, cols_est, "estaticos")
i_pD = cols_dec.index("es_esios_D")        # el indice se ha desplazado: recalcular

class Escalador:
    def fit(self, x):
        ejes = tuple(range(x.ndim - 1))
        self.mu = x.mean(axis=ejes, keepdims=True)
        sd = x.std(axis=ejes, keepdims=True)
        self.sd = np.where(sd < 1e-8, 1.0, sd)      # nunca dividir por ~0
        return self
    def __call__(self, x):
        # clip a 10 sigmas: un valor mas extremo es cambio de regimen o outlier, y en ningun
        # caso debe dominar el gradiente
        return np.clip((x - self.mu) / self.sd, -10, 10).astype("float32")

s_enc = Escalador().fit(X_enc[tr]); s_dec = Escalador().fit(X_dec[tr]); s_est = Escalador().fit(X_est[tr])
mu_y, sd_y = float(y[tr].mean()), float(y[tr].std())   # un solo mu/sd: las 24 salidas
                                                       # comparten unidad (EUR/MWh)
Xe, Xd, Xs = s_enc(X_enc), s_dec(X_dec), s_est(X_est)
ys = ((y - mu_y) / sd_y).astype("float32")
inv = lambda p: p * sd_y + mu_y

# VERIFICACION -- sin esto, un fallo de escalado vuelve a pasar desapercibido
print("\ncomprobacion de escalado (|max| debe quedar bajo el clip, medias de train cerca de 0)")
for n, a in [("Xe", Xe), ("Xd", Xd), ("Xs", Xs)]:
    for m, nm in [(tr, "train"), (va, "val"), (te, "test")]:
        print(f"  {n} {nm:<6} |max| {np.abs(a[m]).max():7.2f}   media {a[m].mean():+6.3f}")
print(f"\nprecio train: media {mu_y:.1f}  sd {sd_y:.1f} EUR/MWh")

## 4. Baselines — obligatorios y primeros

El precio eléctrico tiene una autocorrelación enorme: repetir el precio de ayer ya acierta
bastante. Un MAE de 8 €/MWh no significa nada por sí solo; significa algo **comparado con el
naive**. Sin esta referencia el capítulo no tiene contra qué medirse.

Sirven además de detector de fugas: si una red baja mucho del naive, lo probable no es que sea
brillante sino que se ha colado información posterior al cierre del mercado.

In [ ]:
def metricas(yt, yp):
    e = yp - yt
    den = (np.abs(yt) + np.abs(yp)) / 2          # sMAPE, no MAPE: el precio pasa por cero
    n = np.arange(len(yt))
    # captura de spread: que fraccion del arbitraje perfecto se obtiene cargando en la hora de
    # valle PREDICHA y descargando en la de pico PREDICHA. El MAE mide precision de precio;
    # esto mide valor economico, y no son lo mismo -- un modelo con peor MAE que clave las
    # horas vale mas para operar una bateria que uno preciso que las confunda.
    cap = ((yt[n, yp.argmax(1)] - yt[n, yp.argmin(1)]) /
           np.maximum(yt.max(1) - yt.min(1), .1))
    return {"MAE": float(np.abs(e).mean()),
            "RMSE": float(np.sqrt((e ** 2).mean())),
            "sMAPE": float(100 * np.mean(np.abs(e) / np.maximum(den, 1e-3))),
            "captura_%": float(100 * cap.mean()),
            "pico_1h_%": float(100 * (np.abs(yt.argmax(1) - yp.argmax(1)) <= 1).mean())}

res, preds = {}, {}
i_pD = cols_dec.index("es_esios_D")
res["naive D (precio de hoy)"] = metricas(y[te], X_dec[te][:, :, i_pD])

y_sem = np.stack([y[max(i - 7, 0)] for i in np.where(te)[0]])
res["naive D-6 (misma semana)"] = metricas(y[te], y_sem)

pd.DataFrame(res).T.round(2)

## 4b. Series temporales clásicas — Holt-Winters y SARIMA

El módulo 06.4 del máster fija una metodología para esto y hasta ahora el proyecto no la
aplicaba: los "baselines estadísticos" eran persistencia y media móvil, que son reglas, no
modelos de serie temporal. Aquí entran los dos que faltaban, con el mismo guion del módulo —
descomposición, ACF/PACF, identificación de órdenes, diagnóstico de residuos.

**Cómo se comparan de forma justa.** Un modelo univariante que prediga 576 días seguidos no
dice nada: hay que darle el mismo horizonte que a las redes, un día por delante. Así que se
estiman los parámetros **sólo sobre train** y luego la serie se extiende sin reajustar
(`append(..., refit=False)`), tomando las predicciones a un paso. El modelo ve el pasado real
igual que lo ve la red, y nunca ve el futuro.

**Un modelo por hora.** El precio de las 3:00 y el de las 20:00 son procesos distintos —
distinta estacionalidad semanal, distinta varianza — así que se ajustan 24 series diarias
independientes, que es como el módulo trata una serie con estacionalidad. La estacionalidad
es semanal (s=7), no diaria: dentro de cada serie el paso ya es un día.

Ojo con lo que estos modelos NO pueden hacer, porque es la mitad del argumento: sólo ven el
precio. No saben que mañana hará viento ni que el gas ha subido. Su error es la referencia
que mide cuánto aporta toda la información exógena de las otras 113 columnas.

In [ ]:
import warnings as _w
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox

_w.filterwarnings("ignore")
ESTACION = 7          # semanal: cada serie es un dia, el ciclo que queda es el de la semana

# ── identificacion de ordenes sobre una hora representativa ───────────────────
# El modulo pide identificar (p,d,q)(P,D,Q)[s] por ACF/PACF. Se hace sobre la hora 12 --
# suficientemente lejos del valle nocturno y del pico de tarde -- y el orden encontrado se
# reutiliza en las 24. Ajustar un orden distinto por hora sobreajustaria la seleccion.
h_ref, serie_ref = 12, y[tr][:, 12]

fig, ax = plt.subplots(1, 2, figsize=(14, 3.5))
plot_acf(serie_ref, lags=30, ax=ax[0]); ax[0].set_title(f"ACF · precio de las {h_ref}:00 (train)")
plot_pacf(serie_ref, lags=30, ax=ax[1], method="ywm"); ax[1].set_title("PACF")
plt.tight_layout(); plt.show()

try:
    from pmdarima import auto_arima
    aa = auto_arima(serie_ref, seasonal=True, m=ESTACION, d=None, D=None,
                    max_p=3, max_q=3, max_P=2, max_Q=2, stepwise=True,
                    suppress_warnings=True, error_action="ignore")
    ORDEN, ORDEN_EST = aa.order, aa.seasonal_order
    print(f"auto_arima sobre la hora {h_ref}: SARIMA{ORDEN}{ORDEN_EST}[{ESTACION}]")
except Exception as e:
    ORDEN, ORDEN_EST = (1, 1, 1), (1, 0, 1, ESTACION)
    print(f"auto_arima no disponible ({type(e).__name__}); se usa SARIMA{ORDEN}{ORDEN_EST}")

In [ ]:
# ── ajuste por hora y prediccion a un paso ────────────────────────────────────
def _un_paso(serie, n_train, ajustar):
    """Parametros estimados en train, predicciones a un paso sobre el resto.

    `ajustar` recibe la serie de train y devuelve un objeto con `.append(resto, refit=False)`
    o, para Holt-Winters, se reconstruye con los parametros fijos. En ninguno de los dos
    casos el modelo reestima con datos posteriores a train.
    """
    return ajustar(serie, n_train)

def _hw(serie, n_train):
    m = ExponentialSmoothing(serie[:n_train], trend="add", seasonal="add",
                             seasonal_periods=ESTACION,
                             initialization_method="estimated").fit()
    # misma parametrizacion, ahora sobre la serie entera y SIN reoptimizar: los valores
    # ajustados son predicciones a un paso con los parametros de train
    m2 = ExponentialSmoothing(serie, trend="add", seasonal="add",
                              seasonal_periods=ESTACION,
                              initialization_method="known",
                              initial_level=m.params["initial_level"],
                              initial_trend=m.params["initial_trend"],
                              initial_seasonal=m.params["initial_seasons"]).fit(
        smoothing_level=m.params["smoothing_level"],
        smoothing_trend=m.params["smoothing_trend"],
        smoothing_seasonal=m.params["smoothing_seasonal"], optimized=False)
    return np.asarray(m2.fittedvalues)

def _sarima(serie, n_train):
    r = SARIMAX(serie[:n_train], order=ORDEN, seasonal_order=ORDEN_EST,
                enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
    r2 = r.append(serie[n_train:], refit=False)     # extiende el estado, NO reestima
    return np.asarray(r2.fittedvalues), r

n_tr = int(tr.sum())
pred_hw = np.zeros_like(y)
pred_sa = np.zeros_like(y)
residuos_ref = None
for h in range(24):
    s = y[:, h].astype(float)
    try:
        pred_hw[:, h] = _hw(s, n_tr)
    except Exception as e:
        pred_hw[:, h] = np.roll(s, 1); print(f"  HW hora {h} fallo ({type(e).__name__})")
    try:
        p, r = _sarima(s, n_tr)
        pred_sa[:, h] = p
        if h == h_ref:
            residuos_ref = r.resid
    except Exception as e:
        pred_sa[:, h] = np.roll(s, 1); print(f"  SARIMA hora {h} fallo ({type(e).__name__})")

for nom, p in [("Holt-Winters (s=7)", pred_hw), ("SARIMA (s=7)", pred_sa)]:
    preds[nom] = p[te]
    res[nom] = metricas(y[te], p[te])

print(f"Holt-Winters  MAE {res['Holt-Winters (s=7)']['MAE']:6.2f}")
print(f"SARIMA        MAE {res['SARIMA (s=7)']['MAE']:6.2f}")
print(f"naive D       MAE {res['naive D (precio de hoy)']['MAE']:6.2f}   <- el liston")

In [ ]:
# ── diagnostico de residuos ───────────────────────────────────────────────────
# El modulo lo exige y tiene sentido: si los residuos siguen autocorrelados, el modelo ha
# dejado estructura sin explicar y los ordenes elegidos no son suficientes.
if residuos_ref is not None:
    r = np.asarray(residuos_ref)[ESTACION:]        # los primeros arrastran la inicializacion
    lb = acorr_ljungbox(r, lags=[7, 14, 21], return_df=True)
    print(f"Ljung-Box sobre los residuos de la hora {h_ref}:")
    print(lb.round(4).to_string())
    print()
    if (lb["lb_pvalue"] < 0.05).any():
        print("p < 0,05: queda autocorrelacion sin explicar. Es lo esperable aqui -- el")
        print("precio depende de viento, gas y demanda, y un modelo univariante no los ve.")
    else:
        print("residuos sin autocorrelacion significativa: los ordenes capturan la estructura.")

    fig, ax = plt.subplots(1, 3, figsize=(15, 3.2))
    ax[0].plot(r, lw=0.6); ax[0].set_title("residuos"); ax[0].axhline(0, color="k", lw=0.5)
    plot_acf(r, lags=25, ax=ax[1]); ax[1].set_title("ACF de los residuos")
    ax[2].hist(r, bins=60); ax[2].set_title("distribución")
    plt.tight_layout(); plt.show()

## 5. Modelo 1 — MLP multi-salida

El punto de partida: aplanar todo y una red densa. Sin recurrencia, pero es un competidor serio
y establece si la estructura temporal aporta algo o no.

Frente a `First_Model` de clase, cambia solo el final: `Dense(24)` **lineal** en lugar de
`softmax`, y `Huber` en lugar de crossentropy.

In [ ]:
plano = np.concatenate([Xe.reshape(len(Xe), -1), Xd.reshape(len(Xd), -1), Xs], axis=1)
print("matriz plana (redes):", plano.shape)

# ── vista para modelos de arbol ───────────────────────────────────────────────
# Un arbol NO puede aprovechar los 168 pasos crudos por canal. Las redes si: la LSTM
# comparte pesos entre los pasos, asi que aprende una vez "como se comporta la eolica" y lo
# aplica a los 168. LightGBM, en cambio, ve 168 x C columnas casi identicas entre si y tiene
# que evaluarlas todas en cada nodo para descubrir que importa la de hace 37 horas.
#
# Con 52 canales eso son 8.736 columnas solo de encoder, y como el coste es lineal en
# columnas, los 24 modelos (uno por hora) tardaban ~23 minutos. Resumiendo el encoder a unos
# pocos estadisticos por canal baja a ~800 columnas: mas rapido y con menos ruido donde
# elegir el corte.
#
# Se conservan enteras las 24 horas del ULTIMO dia del precio, que son el ancla del naive y
# lo unico del encoder que el arbol usa hora a hora.
def vista_arbol(Xe, Xd, Xs, ventana=VENTANA_DIAS):
    v = Xe.reshape(len(Xe), ventana, 24, Xe.shape[-1])          # (n, dias, 24, canales)
    partes = [
        v[:, -1, :, 0],                                          # precio del dia D, 24 h
        v.mean(axis=(1, 2)), v.min(axis=(1, 2)), v.max(axis=(1, 2)),
        v[:, -1].mean(axis=1),                                   # media del dia D
        v[:, -1].mean(axis=1) - v[:, 0].mean(axis=1),            # tendencia en la ventana
        Xd.reshape(len(Xd), -1), Xs,
    ]
    return np.concatenate(partes, axis=1).astype("float32")

arbol = vista_arbol(Xe, Xd, Xs)
print(f"matriz plana (arboles): {arbol.shape}  "
      f"-- {plano.shape[1] / arbol.shape[1]:.0f}x mas estrecha que la de redes")

def compilar(m, lr=1e-3):
    m.compile(optimizer=keras.optimizers.Adam(lr),
              loss=keras.losses.Huber(delta=1.0), metrics=["mae"])
    return m

CB = lambda: [keras.callbacks.EarlyStopping(monitor="val_loss", patience=12,
                                            restore_best_weights=True),
              keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5,
                                                min_lr=1e-5)]

mlp = compilar(keras.Sequential([
    layers.Input(shape=(plano.shape[1],)),
    layers.Dense(512, activation="relu"), layers.Dropout(0.2),
    layers.Dense(256, activation="relu"), layers.Dropout(0.2),
    layers.Dense(24),                       # LINEAL: es regresion
], name="MLP"))
mlp.summary()

h_mlp = mlp.fit(plano[tr], ys[tr], validation_data=(plano[va], ys[va]),
                epochs=EPOCHS, batch_size=BATCH, callbacks=CB(), verbose=2)
preds["MLP"] = inv(mlp.predict(plano[te], verbose=0))
res["MLP"] = metricas(y[te], preds["MLP"])
res["MLP"]

## 6. Modelo 2 — LSTM apilada sobre el histórico

Patrón *Deep RNN* de `IMBD_RNN`: dos LSTM encadenadas, la primera con `return_sequences=True`.
El encoder digiere las 168 horas de precio y su salida se concatena con las exógenas aplanadas.

`Bidirectional` **sí** aquí: el encoder solo ve pasado ya ocurrido.

In [ ]:
in_e = keras.Input(shape=Xe.shape[1:], name="hist")
in_d = keras.Input(shape=Xd.shape[1:], name="fut")
in_s = keras.Input(shape=(Xs.shape[1],), name="est")

x = layers.Bidirectional(layers.LSTM(64, return_sequences=True, dropout=0.2))(in_e)
x = layers.Bidirectional(layers.LSTM(64, dropout=0.2))(x)
x = layers.Concatenate()([x, layers.Flatten()(in_d), in_s])
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.2)(x)
out = layers.Dense(24)(x)

lstm = compilar(keras.Model([in_e, in_d, in_s], out, name="LSTM_apilada"))
lstm.summary()

ent = lambda m: {"hist": Xe[m], "fut": Xd[m], "est": Xs[m]}
h_lstm = lstm.fit(ent(tr), ys[tr], validation_data=(ent(va), ys[va]),
                  epochs=EPOCHS, batch_size=BATCH, callbacks=CB(), verbose=2)
preds["LSTM apilada"] = inv(lstm.predict(ent(te), verbose=0))
res["LSTM apilada"] = metricas(y[te], preds["LSTM apilada"])
res["LSTM apilada"]

## 7. Modelo 3 — Seq2Seq encoder-decoder

La arquitectura que encaja con la estructura del problema, y el aporte propio del trabajo.

El esqueleto viene de `Seq2seq` y del notebook de traducción:
`LSTM → RepeatVector → LSTM(return_sequences=True) → TimeDistributed(Dense)`.

**Con una modificación decisiva.** En traducción, `RepeatVector` repite el contexto porque no hay
nada específico de cada paso de salida. Aquí sí lo hay: se conoce la previsión solar *de las
14:00*, la eólica *de las 03:00*, la NTC *de cada hora*. Por eso el contexto repetido se
**concatena** con `X_dec` antes del decoder, en lugar de entrar solo.

Esa concatenación es lo que justifica la arquitectura frente a un modelo tabular, y es
precisamente lo que ningún notebook de clase muestra: en traducción no existe un exógeno futuro
conocido.

El decoder es **unidireccional**, sin excepción.

In [ ]:
in_e = keras.Input(shape=Xe.shape[1:], name="hist")
in_d = keras.Input(shape=Xd.shape[1:], name="fut")
in_s = keras.Input(shape=(Xs.shape[1],), name="est")

# ENCODER: bidireccional (pasado completo, sin fuga)
enc = layers.Bidirectional(layers.LSTM(128, dropout=0.2), name="encoder")(in_e)
ctx = layers.Concatenate(name="contexto")([enc, layers.Dense(64, activation="relu")(in_s)])

# el contexto repetido 24 veces SE CONCATENA con la prevision horaria de D+1
ctx_rep = layers.RepeatVector(24)(ctx)
dec_in = layers.Concatenate(name="contexto_mas_prevision")([ctx_rep, in_d])

# DECODER: unidireccional obligatorio
dec = layers.LSTM(128, return_sequences=True, dropout=0.2, name="decoder")(dec_in)
dec = layers.Dropout(0.2)(dec)
out = layers.Reshape((24,))(layers.TimeDistributed(layers.Dense(1))(dec))

s2s = compilar(keras.Model([in_e, in_d, in_s], out, name="Seq2Seq"))
s2s.summary()

h_s2s = s2s.fit(ent(tr), ys[tr], validation_data=(ent(va), ys[va]),
                epochs=EPOCHS, batch_size=BATCH, callbacks=CB(), verbose=2)
preds["Seq2Seq"] = inv(s2s.predict(ent(te), verbose=0))
res["Seq2Seq"] = metricas(y[te], preds["Seq2Seq"])
res["Seq2Seq"]

## 8. Comparativa y análisis del error

In [ ]:
tabla = pd.DataFrame(res).T.sort_values("MAE")
base = tabla.loc[[i for i in tabla.index if i.startswith("naive")], "MAE"].min()
tabla["vs_naive_%"] = (100 * (tabla["MAE"] / base - 1)).round(1)
display(tabla.round(2))

mejor = tabla.index[0]
print(f"\nMejor modelo: {mejor}")
if tabla.loc[mejor, "MAE"] < 3:
    print("*** MAE sospechosamente bajo: antes de darlo por bueno, revisa si algun canal de")
    print("    X_dec contiene informacion posterior al cierre del mercado. ***")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))

for k, h in [("MLP", h_mlp), ("LSTM apilada", h_lstm), ("Seq2Seq", h_s2s)]:
    ax[0].plot(h.history["val_loss"], label=k)
ax[0].set_title("Pérdida en validación (Huber)"); ax[0].set_xlabel("época"); ax[0].legend()

for k in preds:
    ax[1].plot(np.abs(preds[k] - y[te]).mean(axis=0), marker="o", ms=3, label=k)
ax[1].plot(np.abs(X_dec[te][:, :, i_pD] - y[te]).mean(axis=0), "k--", label="naive D")
ax[1].set_title("MAE por hora del día"); ax[1].set_xlabel("hora"); ax[1].set_ylabel("€/MWh")
ax[1].legend(); plt.tight_layout(); plt.show()

In [ ]:
# Los baselines tambien son predicciones: si el mejor resulta ser uno de ellos, tiene que poder
# graficarse igual que las redes (esto provocaba un KeyError en la version anterior).
preds["naive D (precio de hoy)"] = X_dec[te][:, :, i_pD]
preds["naive D-6 (misma semana)"] = y_sem

err_dia = np.abs(preds[mejor] - y[te]).mean(axis=1)
orden = np.argsort(err_dia)
sel = [int(orden[0]), int(orden[len(orden) // 2]), int(orden[-1])]
titulos = ["mejor día", "día mediano", "peor día"]

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5), sharey=True)
for a, i, t in zip(axes, sel, titulos):
    a.plot(y[te][i], "o-", color="black", label="real")
    a.plot(preds[mejor][i], "s--", label=mejor)
    a.set_title(f"{t} — {pd.Timestamp(fechas[te][i]).date()}  MAE {err_dia[i]:.1f}")
    a.set_xlabel("hora")
axes[0].set_ylabel("€/MWh"); axes[0].legend(); plt.tight_layout(); plt.show()

## 8b. Diagnóstico visual de un modelo

Los gráficos aquí no son adorno: son el diagnóstico. Con un MAE alto, la curva dice en dos
segundos si el modelo predice plano, si va sesgado en nivel o si solo falla en ciertas horas —
y cada caso tiene un remedio distinto.

**Cómo leer cada panel**

- **Dispersión.** El más informativo. Diagonal ancha = funciona, solo hay ruido. Nube *aplanada
  en horizontal* (predice ~90 tanto si el real es 20 como si es 200) = el modelo ha colapsado a
  la media del train.
- **Histograma del error.** Centrado en cero es ruido; desplazado es sesgo de nivel, y el
  remedio es la §8c.
- **MAE por hora.** Si pierde contra el naive solo en las horas de rampa solar (8–10, 18–20) es
  un problema de *forma*; si pierde en todas por igual, es de *nivel*.
- **Peor día.** Mira si acierta al menos la forma aunque falle la escala. Una línea plana
  significa que no ha aprendido nada.

In [ ]:
MODELO = "Seq2Seq"          # cambia aqui el modelo a inspeccionar
MASCARA, NOMBRE_M = te, "test"   # durante la iteracion usa `va`; ver nota de la seccion 9

p, yt = preds[MODELO], y[MASCARA]
fm = pd.to_datetime(fechas[MASCARA])
ts = np.repeat(fm, 24) + pd.to_timedelta(np.tile(np.arange(24), len(fm)), "h")
err = p - yt

fig = plt.figure(figsize=(15, 9))

ax = fig.add_subplot(3, 1, 1)
ax.plot(ts, yt.ravel(), lw=.7, color="black", label="real")
ax.plot(ts, p.ravel(), lw=.7, alpha=.8, label=MODELO)
ax.set_title(f"{MODELO} — {NOMBRE_M} completo   MAE {np.abs(err).mean():.2f} €/MWh")
ax.set_ylabel("€/MWh"); ax.legend()

ax = fig.add_subplot(3, 3, 4)
ax.scatter(yt.ravel(), p.ravel(), s=2, alpha=.15)
lim = [float(min(yt.min(), p.min())), float(max(yt.max(), p.max()))]
ax.plot(lim, lim, "r--", lw=1)
ax.set_xlabel("real"); ax.set_ylabel("predicho"); ax.set_title("Real vs predicho")

ax = fig.add_subplot(3, 3, 5)
ax.hist(err.ravel(), bins=60, color="steelblue")
ax.axvline(0, color="r", ls="--")
ax.set_title(f"Error   sesgo {err.mean():+.2f} €/MWh")

ax = fig.add_subplot(3, 3, 6)
ax.plot(np.abs(err).mean(axis=0), "o-", label=MODELO)
ax.plot(np.abs(X_dec[MASCARA][:, :, i_pD] - yt).mean(axis=0), "k--", label="naive D")
ax.set_xlabel("hora"); ax.set_ylabel("MAE"); ax.set_title("Error por hora"); ax.legend()

e_dia = np.abs(err).mean(axis=1); orden = np.argsort(e_dia)
for k, (i, t) in enumerate(zip([orden[0], orden[len(orden)//2], orden[-1]],
                               ["mejor día", "día mediano", "peor día"])):
    ax = fig.add_subplot(3, 3, 7 + k)
    ax.plot(yt[i], "o-", color="black", label="real")
    ax.plot(p[i], "s--", label=MODELO)
    ax.plot(X_dec[MASCARA][i, :, i_pD], ":", color="grey", label="naive")
    ax.set_title(f"{t} — {fm[i].date()}  MAE {e_dia[i]:.1f}")
    ax.set_xlabel("hora")
    if k == 0: ax.legend(fontsize=8)

plt.tight_layout(); plt.show()

## 8c. Variante: predecir el residuo frente al naive

Si el histograma de la §8b sale desplazado, el modelo ha aprendido el **nivel** de precios del
entrenamiento y lo arrastra. Es esperable: train incluye la crisis del gas de 2022 (media de 93
€/MWh) y el test es 2026, con niveles muy distintos. Una red regularizada tira hacia la media
del target que ha visto; el naive es inmune porque no aprende ningún nivel, solo copia.

**El remedio es cambiar el target, no la arquitectura.** En vez de aprender cuánto vale el
precio de mañana, la red aprende *cuánto se desvía respecto a hoy*. El target se vuelve
estacionario, el cambio de régimen deja de importar, y de regalo el modelo parte empatando al
naive: solo tiene que mejorarlo.

Compara el `sd` del residuo con el del precio: ahí se ve la ganancia. El modelo deja de tener
que reproducir un rango de 0 a 500 y solo modela un margen estrecho.

In [ ]:
naive_todos = X_dec[:, :, i_pD]                  # precio de D por hora, escala original
resid = y - naive_todos
mu_r, sd_r = float(resid[tr].mean()), float(resid[tr].std())
yr = ((resid - mu_r) / sd_r).astype("float32")
inv_r = lambda p, m: p * sd_r + mu_r + naive_todos[m]

print(f"residuo train: media {mu_r:+.2f}  sd {sd_r:.2f} EUR/MWh")
print(f"precio  train: media {mu_y:+.2f}  sd {sd_y:.2f} EUR/MWh   <- cuanto menor el sd del")
print("                                                              residuo, mas facil el problema")

def seq2seq(u=96, dr=0.3):
    "Misma arquitectura de la seccion 7, con menos capacidad: 1.778 dias no dan para 128 unidades."
    ie = keras.Input(shape=Xe.shape[1:], name="hist")
    idc = keras.Input(shape=Xd.shape[1:], name="fut")
    ist = keras.Input(shape=(Xs.shape[1],), name="est")
    enc = layers.Bidirectional(layers.LSTM(u, dropout=dr))(ie)
    ctx = layers.Concatenate()([enc, layers.Dense(64, activation="relu")(ist)])
    dec = layers.Concatenate()([layers.RepeatVector(24)(ctx), idc])
    dec = layers.LSTM(u, return_sequences=True, dropout=dr)(dec)
    out = layers.Reshape((24,))(layers.TimeDistributed(layers.Dense(1))(dec))
    return compilar(keras.Model([ie, idc, ist], out, name="Seq2Seq_residuo"))

s2r = seq2seq()
h_s2r = s2r.fit(ent(tr), yr[tr], validation_data=(ent(va), yr[va]),
                epochs=EPOCHS, batch_size=BATCH, callbacks=CB(), verbose=2)

preds["Seq2Seq residuo"] = inv_r(s2r.predict(ent(te), verbose=0), te)
res["Seq2Seq residuo"] = metricas(y[te], preds["Seq2Seq residuo"])

tabla = pd.DataFrame(res).T.sort_values("MAE")
base = tabla.loc[[i for i in tabla.index if i.startswith("naive")], "MAE"].min()
tabla["vs_naive_%"] = (100 * (tabla["MAE"] / base - 1)).round(1)
display(tabla.round(2))

## 8d. Residuo aplicado al resto de modelos, y ensemble

El residuo funcionó en el Seq2Seq (MAE 15,14 frente a los 17,13 del naive). Toca aplicarlo a
todo: el MLP en escala absoluta ya iba mejor que el Seq2Seq (20,89 vs 27,20), así que en residuo
debería quedar por debajo de 15.

Se baja además la capacidad (256/128 en vez de 512/256, dropout 0,35 en vez de 0,2): con 1.778
días de entrenamiento y `loss 0.15` frente a `val_loss 0.25`, sobra red.

In [ ]:
mlp_r = compilar(keras.Sequential([
    layers.Input(shape=(plano.shape[1],)),
    layers.Dense(256, activation="relu"), layers.Dropout(0.35),
    layers.Dense(128, activation="relu"), layers.Dropout(0.35),
    layers.Dense(24),
], name="MLP_residuo"))
h_mlpr = mlp_r.fit(plano[tr], yr[tr], validation_data=(plano[va], yr[va]),
                   epochs=EPOCHS, batch_size=BATCH, callbacks=CB(), verbose=0)
preds["MLP residuo"] = inv_r(mlp_r.predict(plano[te], verbose=0), te)
res["MLP residuo"] = metricas(y[te], preds["MLP residuo"])
print("MLP residuo:", {k: round(float(v), 2) for k, v in res["MLP residuo"].items()})

In [ ]:
# LightGBM sobre el residuo: 24 modelos, uno por hora. Es el competidor serio del deep learning
# en tabular con pocos miles de observaciones -- si gana, se reporta tal cual.
try:
    import lightgbm as lgb
    pl = np.zeros((int(te.sum()), 24))
    for h in range(24):
        g = lgb.LGBMRegressor(n_estimators=800, learning_rate=0.03, num_leaves=31,
                              objective="huber", verbose=-1, random_state=SEMILLA,
                              n_jobs=-1)
        g.fit(arbol[tr], yr[tr][:, h], eval_set=[(arbol[va], yr[va][:, h])],
              callbacks=[lgb.early_stopping(50, verbose=False)])
        pl[:, h] = g.predict(arbol[te])
    preds["LightGBM residuo"] = inv_r(pl, te)
    res["LightGBM residuo"] = metricas(y[te], preds["LightGBM residuo"])
    print("LightGBM residuo:", {k: round(float(v), 2) for k, v in res["LightGBM residuo"].items()})
except ImportError:
    print("lightgbm no instalado (pip install lightgbm)")

## 8e. La familia recurrente completa — SimpleRNN, GRU, LSTM y CNN-LSTM

El módulo 14 construye estas cuatro y pide compararlas explícitamente ("cambia `SimpleRNN`
por `LSTM` y por `GRU`"). Van sobre el **residuo frente al naive**, que la sección 8c deja demostrado como la formulación que funciona, y comparten el **mismo esqueleto** — mismo encoder-decoder,
mismos datos, misma semilla — cambiando sólo la celda recurrente, que es lo que hace que la
comparación mida la arquitectura y no otra cosa.

| variante | qué añade |
|---|---|
| `SimpleRNN` | el escalón básico: sin puertas, sufre el gradiente que se desvanece a 168 pasos |
| `GRU` | dos puertas y un tercio menos de parámetros que la LSTM — con 2.388 muestras puede ganar |
| `LSTM` | tres puertas, la referencia |
| `Conv1D+LSTM` | una convolución con stride reduce los 168 pasos antes de la recurrente |

La última merece explicación. El encoder tiene 168 pasos × 114 canales, y una LSTM tiene que
recorrerlos uno a uno. Un `Conv1D` con stride 4 los comprime a 42 extrayendo el patrón local
—la rampa solar, el pico de tarde— y deja a la recurrente la dinámica de medio plazo. Es el
CNN-LSTM clásico: suele igualar o mejorar, y entrena bastante más rápido, que con 40 minutos
por bloque en Colab no es un detalle menor.

In [ ]:
CELDAS = {"SimpleRNN": layers.SimpleRNN, "GRU": layers.GRU, "LSTM": layers.LSTM}

def recurrente(celda="LSTM", u=128, dr=0.2, conv=None, bidireccional=True):
    """Encoder-decoder con la celda recurrente intercambiable.

    `conv` = (filtros, kernel, stride) antepone un Conv1D al encoder. `bidireccional` sólo
    afecta al encoder: el decoder tiene que ser unidireccional porque genera las 24 horas
    en orden y no puede mirar hacia adelante.
    """
    C = CELDAS[celda]
    in_e = keras.Input(shape=Xe.shape[1:], name="hist")
    in_d = keras.Input(shape=Xd.shape[1:], name="fut")
    in_s = keras.Input(shape=(Xs.shape[1],), name="est")

    x = in_e
    if conv:
        f, k, s = conv
        x = layers.Conv1D(f, k, strides=s, padding="causal", activation="relu",
                          name="conv_local")(x)
        x = layers.BatchNormalization()(x)

    nucleo = C(u, dropout=dr)
    enc = layers.Bidirectional(nucleo, name="encoder")(x) if bidireccional else nucleo(x)
    ctx = layers.Concatenate()([enc, layers.Dense(64, activation="relu")(in_s)])
    dec_in = layers.Concatenate()([layers.RepeatVector(24)(ctx), in_d])
    dec = C(u, return_sequences=True, dropout=dr, name="decoder")(dec_in)
    out = layers.Reshape((24,))(layers.TimeDistributed(layers.Dense(1))(dec))

    nombre = celda + ("_conv" if conv else "")
    return compilar(keras.Model([in_e, in_d, in_s], out, name=nombre))

VARIANTES = [("SimpleRNN", dict(celda="SimpleRNN", u=64)),
             ("GRU",       dict(celda="GRU")),
             ("LSTM",      dict(celda="LSTM")),
             ("Conv1D+LSTM", dict(celda="LSTM", conv=(64, 5, 4)))]

comparativa_fam = []
for nombre, kw in VARIANTES:
    keras.utils.set_random_seed(SEMILLA)              # misma inicializacion para todas
    m = recurrente(**kw)
    m.fit(ent(tr), yr[tr], validation_data=(ent(va), yr[va]),
          epochs=EPOCHS, batch_size=BATCH, callbacks=CB(), verbose=0)
    p_va = inv_r(m.predict(ent(va), verbose=0), va)
    p_te = inv_r(m.predict(ent(te), verbose=0), te)
    etiqueta = f"{nombre} residuo"
    preds[etiqueta] = p_te
    res[etiqueta] = metricas(y[te], p_te)
    comparativa_fam.append({"variante": nombre, "parametros": m.count_params(),
                            "MAE_val": round(float(metricas(y[va], p_va)["MAE"]), 2),
                            "MAE_test": round(float(res[etiqueta]["MAE"]), 2)})
    print(f"  {nombre:14s} {m.count_params():>9,} par · "
          f"MAE val {comparativa_fam[-1]['MAE_val']:5.2f} · test {comparativa_fam[-1]['MAE_test']:5.2f}")

display(pd.DataFrame(comparativa_fam).sort_values("MAE_val"))
print("Se elige por MAE de VALIDACION. El de test se muestra para ver si el orden aguanta,")
print("no para decidir con el.")

## 8f. Lo que faltaba del temario — estados, L2 y atención

Repasando los notebooks de clase del módulo 14 y de deep learning, hay cuatro técnicas que
se usan allí y que este notebook no aplicaba. Tres son mejoras plausibles y una corrige una
diferencia arquitectónica real.

**El Seq2Seq no era un Seq2Seq.** El de la sección 7 comprime el encoder a un vector y lo
repite 24 veces con `RepeatVector`. El canónico —el que construye `Seq2seq.ipynb` en clase—
pasa los **estados** del encoder al decoder:

```python
_, state_h, state_c = layers.LSTM(64, return_state=True)(encoder_input)
decoder_output = layers.LSTM(64, return_sequences=True)(decoder_input,
                                                        initial_state=[state_h, state_c])
```

La diferencia no es cosmética. Con `RepeatVector`, el decoder recibe el mismo contexto
pegado a cada una de las 24 horas y arranca su memoria en cero. Con `initial_state`, arranca
**con la memoria del encoder ya cargada** y la va consumiendo hora a hora, que es lo que un
decoder recurrente debe hacer. `return_state` aparece 9 veces en los notebooks de clase.

**Regularización L2.** `kernel_regularizer=l2(...)` es la segunda técnica más usada del
temario —16 apariciones, sólo por detrás de Dropout— y aquí no se usaba ninguna vez. Y hace
falta: con 2.388 ejemplos de entrenamiento y modelos de millones de parámetros, la `val_loss`
rebotaba entre 0,065 y 0,116 sin asentarse, que es la firma del sobreajuste. Dropout apaga
neuronas; L2 encoge pesos. Son complementarias, no alternativas.

**Atención.** Con `initial_state` el decoder recibe un resumen del encoder; con atención
puede además **mirar hacia atrás y elegir qué horas del histórico le importan** para cada
hora que predice. Para el precio eléctrico tiene sentido físico: la hora de pico de mañana
se parece más a la hora de pico de días anteriores que a la madrugada.

**Ruido gaussiano en la entrada.** Regularización barata: perturbar la entrada obliga al
modelo a no depender de valores exactos. Tres apariciones en el temario.

Se prueban **de una en una y acumulándose**, para saber qué aporta cada cosa en lugar de
lanzar todo junto y no poder atribuir la mejora.

In [ ]:
from tensorflow.keras import regularizers

def seq2seq_temario(u=96, dr=0.3, l2=0.0, estados=True, atencion=False, ruido=0.0,
                    bidireccional=True):
    """Encoder-decoder con las piezas del modulo 14, activables una a una.

    `estados`    pasa (h, c) del encoder como `initial_state` del decoder -- el patron
                 canonico de la clase. Con False vuelve al RepeatVector de la seccion 7.
    `atencion`   deja que el decoder consulte la secuencia completa del encoder.
    `l2`         kernel_regularizer en recurrentes y densas.
    `ruido`      GaussianNoise sobre la ventana del encoder.
    """
    reg = regularizers.l2(l2) if l2 else None
    in_e = keras.Input(shape=Xe.shape[1:], name="hist")
    in_d = keras.Input(shape=Xd.shape[1:], name="fut")
    in_s = keras.Input(shape=(Xs.shape[1],), name="est")

    x = layers.GaussianNoise(ruido)(in_e) if ruido else in_e

    # ENCODER. Devuelve la secuencia (para la atencion) y los estados (para el decoder).
    celda = layers.LSTM(u, return_sequences=True, return_state=True, dropout=dr,
                        kernel_regularizer=reg, name="encoder")
    if bidireccional:
        seq_e, fh, fc, bh, bc = layers.Bidirectional(celda, name="bi_encoder")(x)
        # los estados vienen duplicados (ida y vuelta): se proyectan de vuelta a `u` para
        # que el decoder no tenga que doblar de tamano solo por ser bidireccional
        h = layers.Dense(u, activation="tanh", kernel_regularizer=reg)(
            layers.Concatenate()([fh, bh]))
        c = layers.Dense(u, activation="tanh", kernel_regularizer=reg)(
            layers.Concatenate()([fc, bc]))
    else:
        seq_e, h, c = celda(x)

    ctx = layers.Dense(64, activation="relu", kernel_regularizer=reg)(in_s)
    dec_in = layers.Concatenate(name="entrada_decoder")(
        [in_d, layers.RepeatVector(24)(ctx)])

    if not estados:
        # variante de la seccion 7: el contexto pegado, memoria del decoder a cero
        resumen = layers.Concatenate()([h, c])
        dec_in = layers.Concatenate()([dec_in, layers.RepeatVector(24)(resumen)])
        estado_ini = None
    else:
        estado_ini = [h, c]

    dec = layers.LSTM(u, return_sequences=True, dropout=dr, kernel_regularizer=reg,
                      name="decoder")(dec_in, initial_state=estado_ini)

    if atencion:
        # Cada hora de D+1 consulta las 168 del historico y se queda con lo que le sirve.
        # `Attention` exige que consulta y valores tengan la misma dimension, y el encoder
        # bidireccional devuelve 2u contra las u del decoder: se proyecta paso a paso.
        val = layers.TimeDistributed(
            layers.Dense(u, kernel_regularizer=reg), name="proyeccion_encoder")(seq_e)
        at = layers.Attention(name="atencion")([dec, val])
        dec = layers.Concatenate()([dec, at])

    dec = layers.Dropout(dr)(dec)
    out = layers.Reshape((24,))(
        layers.TimeDistributed(layers.Dense(1, kernel_regularizer=reg))(dec))
    return compilar(keras.Model([in_e, in_d, in_s], out, name="s2s_temario"))


# Acumulativo: cada fila anade una pieza a la anterior, asi que la diferencia entre dos
# filas consecutivas es lo que aporta ESA pieza.
ESCALERA = [
    ("base (RepeatVector, como la sección 7)", dict(estados=False)),
    ("+ estados del encoder",                  dict(estados=True)),
    ("+ regularización L2",                    dict(estados=True, l2=1e-4)),
    ("+ atención",                             dict(estados=True, l2=1e-4, atencion=True)),
    ("+ ruido gaussiano",                      dict(estados=True, l2=1e-4, atencion=True,
                                                    ruido=0.05)),
]

escalera = []
for etiqueta, kw in ESCALERA:
    keras.utils.set_random_seed(SEMILLA)
    m = seq2seq_temario(**kw)
    m.fit(ent(tr), yr[tr], validation_data=(ent(va), yr[va]),
          epochs=EPOCHS, batch_size=BATCH, callbacks=CB(), verbose=0)
    mv = float(metricas(y[va], inv_r(m.predict(ent(va), verbose=0), va))["MAE"])
    p_te = inv_r(m.predict(ent(te), verbose=0), te)
    escalera.append({"variante": etiqueta, "parametros": m.count_params(),
                     "MAE_val": round(mv, 3),
                     "MAE_test": round(float(metricas(y[te], p_te)["MAE"]), 2)})
    print(f"  {etiqueta:42s} {m.count_params():>9,} par · MAE val {mv:6.3f}")
    preds[f"s2s temario: {etiqueta}"] = p_te
    res[f"s2s temario: {etiqueta}"] = metricas(y[te], p_te)

esc = pd.DataFrame(escalera)
esc["aporta"] = (-esc["MAE_val"].diff()).round(3)
display(esc)

mejor = esc.loc[esc.MAE_val.idxmin()]
print()
print(f"Mejor: {mejor['variante']}  (MAE val {mejor['MAE_val']})")
print()
print("La columna `aporta` es la lectura: cuanto baja el MAE de validación al añadir esa")
print("pieza sobre la anterior. Un valor negativo significa que esa técnica ESTORBA aquí,")
print("y eso es tan publicable como lo contrario -- con 2.388 ejemplos no todo lo que")
print("funciona en un dataset grande funciona en este.")

### Ablación por canal — qué aporta de verdad una columna

Quitar una pieza y medir cuánto empeora. Es la única forma honesta de decidir sobre una
columna dudosa, porque **mide error, no asociación**: la V de Cramér mira la variable en
solitario y la correlación parcial sólo descuenta la tendencia lineal, pero ninguna de las
dos ve lo que hace la columna *en compañía de las otras 113*, que es como el modelo la usa.

El caso que la motiva es `bil_direct_consumer_mw_D`, donde los dos estadísticos se
contradicen: V de Cramér **0,282** (muy por encima de la mediana del pool, 0,166) contra una
correlación parcial de **−0,072**, que es nada. Uno dice que importa y el otro que no.

**Se repite con varias semillas.** Una sola ejecución no sirve: la diferencia entre tener y
no tener una columna de 114 es del orden del ruido de inicialización, así que se entrenan
`REPETICIONES` modelos por condición y se compara la media. Sin eso, la ablación mide la
semilla y no la columna.

In [ ]:
CANAL_ABLACION = "bil_direct_consumer_mw_D"
REPETICIONES = 3          # por condicion; con GPU son ~3 min cada modelo

def _sin_canal(nombre):
    """Devuelve (Xe, Xd, Xs) sin ese canal, lo tenga donde lo tenga."""
    base = nombre[:-2] + "@D" if nombre.endswith("_D") else nombre
    for etiqueta in (nombre, base, nombre.replace("_D", "@D")):
        if etiqueta in canales:
            i = canales.index(etiqueta)
            return np.delete(Xe, i, axis=2), Xd, Xs, f"encoder (canal {i})"
    if nombre in cols_dec:
        i = cols_dec.index(nombre)
        return Xe, np.delete(Xd, i, axis=2), Xs, f"decoder (columna {i})"
    if nombre in cols_est:
        i = cols_est.index(nombre)
        return Xe, Xd, np.delete(Xs, i, axis=1), f"estaticos (columna {i})"
    raise KeyError(f"{nombre} no esta en ningun bloque")

Xe_s, Xd_s, Xs_s, donde = _sin_canal(CANAL_ABLACION)
print(f"{CANAL_ABLACION} vive en: {donde}")
print(f"con la columna : Xe {Xe.shape}")
print(f"sin la columna : Xe {Xe_s.shape}")
print()

def _entrena(xe, xd, xs, semilla):
    keras.utils.set_random_seed(semilla)
    in_e = keras.Input(shape=xe.shape[1:]); in_d = keras.Input(shape=xd.shape[1:])
    in_s = keras.Input(shape=(xs.shape[1],))
    enc = layers.Bidirectional(layers.LSTM(96, dropout=0.3))(in_e)
    ctx = layers.Concatenate()([enc, layers.Dense(64, activation="relu")(in_s)])
    dec = layers.LSTM(96, return_sequences=True, dropout=0.3)(
        layers.Concatenate()([layers.RepeatVector(24)(ctx), in_d]))
    out = layers.Reshape((24,))(layers.TimeDistributed(layers.Dense(1))(dec))
    m = compilar(keras.Model([in_e, in_d, in_s], out))
    d = lambda msk: {"input_layer": xe[msk], "input_layer_1": xd[msk], "input_layer_2": xs[msk]}
    m.fit([xe[tr], xd[tr], xs[tr]], yr[tr],
          validation_data=([xe[va], xd[va], xs[va]], yr[va]),
          epochs=EPOCHS, batch_size=BATCH, callbacks=CB(), verbose=0)
    p = m.predict([xe[va], xd[va], xs[va]], verbose=0)
    return float(metricas(y[va], inv_r(p, va))["MAE"])

filas = []
for etiqueta, (xe, xd, xs) in [("con la columna", (Xe, Xd, Xs)),
                               ("sin la columna", (Xe_s, Xd_s, Xs_s))]:
    maes = [_entrena(xe, xd, xs, SEMILLA + k) for k in range(REPETICIONES)]
    filas.append({"condicion": etiqueta, "MAE_val_medio": round(float(np.mean(maes)), 3),
                  "sd": round(float(np.std(maes)), 3),
                  "runs": [round(m, 2) for m in maes]})
    print(f"  {etiqueta:16s} MAE val {np.mean(maes):6.3f} ± {np.std(maes):.3f}   {[round(m,2) for m in maes]}")

abl = pd.DataFrame(filas)
display(abl)
dif = abl.MAE_val_medio[1] - abl.MAE_val_medio[0]
ruido = float(abl.sd.max())
print()
print(f"quitarla cambia el MAE en {dif:+.3f}, y el ruido entre semillas es ±{ruido:.3f}")
if abs(dif) < ruido:
    print(f"La diferencia NO supera al ruido: la columna es indiferente. Se puede quitar")
    print(f"por simplicidad, pero mantenerla tampoco hace daño.")
elif dif > 0:
    print(f"Quitarla EMPEORA: la columna aporta pese a lo que decia la correlacion parcial.")
    print(f"Es la señal de que aportaba en combinacion con otras, no por si sola.")
else:
    print(f"Quitarla MEJORA: la columna metia ruido. Fuera del pool.")

### Fine-tuning por régimen — con el corte medido, no elegido

Preentrenar con toda la historia y rematar con los días recientes. La idea es buena; lo que
estaba mal era el corte, fijado en `2023-06-01` sin criterio.

Y hay un corte que sí tiene fundamento: **2024-04-01**, cuando arranca el archivo de
previsión de ECMWF. Antes de esa fecha el canal meteorológico es pseudo-previsión —ERA5 real
degradado con el error de previsión medido— y a partir de ahí es previsión de verdad. Rematar
el entrenamiento ahí hace que las últimas épocas vean exactamente el tipo de dato que el
modelo recibirá a las 11:00 en producción.

Pero en lugar de sustituir una fecha arbitraria por otra razonada, se **prueban las cuatro** y
se compara en validación. El coste es de una fase corta por corte, y el resultado es un
argumento en vez de una preferencia.

In [ ]:
CORTES_FT = ["2023-01-01", "2023-06-01", "2024-01-01", "2024-04-01"]
ETIQUETA_FT = {"2024-04-01": "arranque de la previsión ECMWF",
               "2024-01-01": "año natural",
               "2023-06-01": "el corte que había",
               "2023-01-01": "ventana de `moderna`"}

# fase 1: una sola vez, y se reutilizan los pesos para todos los cortes
keras.utils.set_random_seed(SEMILLA)
base = seq2seq(u=96, dr=0.3)
base.fit(ent(tr), yr[tr], validation_data=(ent(va), yr[va]),
         epochs=EPOCHS, batch_size=BATCH, callbacks=CB(), verbose=0)
pesos_base = base.get_weights()
mae_base_va = float(metricas(y[va], inv_r(base.predict(ent(va), verbose=0), va))["MAE"])
print(f"fase 1 · histórico completo: MAE val {mae_base_va:.2f}")
print()

ff = pd.to_datetime(fechas)
tabla_ft, mejor = [], (None, mae_base_va, None)
for corte in CORTES_FT:
    reciente = (ff > corte) & (ff <= "2024-12-31")
    if reciente.sum() < 60:
        print(f"  {corte}: solo {reciente.sum()} días, se omite"); continue
    m = keras.models.clone_model(base)
    m.set_weights(pesos_base)
    m.compile(optimizer=keras.optimizers.Adam(1e-4),          # 10x menor que la fase 1
              loss=keras.losses.Huber(delta=1.0), metrics=["mae"])
    m.fit(ent(reciente), yr[reciente], validation_data=(ent(va), yr[va]),
          epochs=40, batch_size=BATCH, verbose=0,
          callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=8,
                                                   restore_best_weights=True)])
    mv = float(metricas(y[va], inv_r(m.predict(ent(va), verbose=0), va))["MAE"])
    tabla_ft.append({"corte": corte, "motivo": ETIQUETA_FT.get(corte, ""),
                     "dias_fase2": int(reciente.sum()), "MAE_val": round(mv, 2),
                     "mejora": round(mae_base_va - mv, 2)})
    print(f"  {corte} ({ETIQUETA_FT.get(corte,''):32s}) {int(reciente.sum()):4d} días · "
          f"MAE val {mv:5.2f}  ({mae_base_va - mv:+.2f})")
    if mv < mejor[1]:
        mejor = (corte, mv, m)

display(pd.DataFrame(tabla_ft).sort_values("MAE_val"))
if mejor[0] is None:
    print("Ningún corte mejora la fase 1: el fine-tuning por régimen no aporta aquí,")
    print("y eso también es un resultado -- significa que el histórico largo no estorba.")
else:
    print(f"Mejor corte: {mejor[0]} ({ETIQUETA_FT.get(mejor[0],'')})")
    preds["Seq2Seq residuo + FT"] = inv_r(mejor[2].predict(ent(te), verbose=0), te)
    res["Seq2Seq residuo + FT"] = metricas(y[te], preds["Seq2Seq residuo + FT"])
    print(f"  MAE test: {res['Seq2Seq residuo + FT']['MAE']:.2f}")

### Ponderación por recencia

Versión suave de lo anterior: en vez de dos fases, un solo entrenamiento donde los días recientes
pesan más. Decaimiento exponencial con vida media de un año.

In [ ]:
antig = (pd.Timestamp("2024-12-31") - pd.to_datetime(fechas[tr])).days.values
w = np.exp(-antig / 550.0).astype("float32")
print(f"peso: dia mas antiguo {w.min():.3f} | mas reciente {w.max():.3f}")

s2r_w = seq2seq(u=96, dr=0.3)
s2r_w.fit(ent(tr), yr[tr], sample_weight=w, validation_data=(ent(va), yr[va]),
          epochs=EPOCHS, batch_size=BATCH, callbacks=CB(), verbose=0)
preds["Seq2Seq residuo + pesos"] = inv_r(s2r_w.predict(ent(te), verbose=0), te)
res["Seq2Seq residuo + pesos"] = metricas(y[te], preds["Seq2Seq residuo + pesos"])
print("con pesos:", {k: round(float(v), 2) for k, v in res["Seq2Seq residuo + pesos"].items()})

### Búsqueda de hiperparámetros — sobre validación y con freno

**Ocho configuraciones y se para.** Con 357 días de validación, probar cincuenta combinaciones y
quedarse con la mejor es sobreajustar la validación: el MAE elegido deja de estimar nada. Es una
limitación real del tamaño de muestra y conviene decirlo en la memoria, no esconderla.

Ojo con `recurrent_dropout` (aparece en `03_deep_learning_ner`): desactiva el kernel optimizado
de LSTM y en CPU multiplica por cinco o más el tiempo por época.

In [ ]:
from itertools import product

registro = []
for u, dr, lr in product([48, 96], [0.25, 0.4], [1e-3, 3e-4]):
    m = seq2seq(u=u, dr=dr)
    m.compile(optimizer=keras.optimizers.Adam(lr),
              loss=keras.losses.Huber(delta=1.0), metrics=["mae"])
    m.fit(ent(tr), yr[tr], validation_data=(ent(va), yr[va]),
          epochs=60, batch_size=BATCH, callbacks=CB(), verbose=0)
    mae_va = metricas(y[va], inv_r(m.predict(ent(va), verbose=0), va))["MAE"]
    registro.append({"unidades": u, "dropout": dr, "lr": lr, "MAE_val": round(float(mae_va), 2)})
    print(registro[-1])

busqueda = pd.DataFrame(registro).sort_values("MAE_val")
busqueda.to_csv("busqueda_hiperparametros.csv", index=False)
display(busqueda)

### Ensemble

Promediar modelos que ya baten al naive suele dar un 3–5% de MAE adicional casi gratis: los
errores de arquitecturas distintas están poco correlacionados y se cancelan en parte.

In [ ]:
base_mae = res["naive D (precio de hoy)"]["MAE"]
buenos = [k for k, v in res.items() if k in preds and v["MAE"] < base_mae]
if len(buenos) > 1:
    preds["Ensemble"] = np.mean([preds[k] for k in buenos], axis=0)
    res["Ensemble"] = metricas(y[te], preds["Ensemble"])
    print("ensemble de:", buenos)

tabla = pd.DataFrame(res).T.sort_values("MAE")
tabla["vs_naive_%"] = (100 * (tabla["MAE"] / base_mae - 1)).round(1)
display(tabla.round(2))
mejor = tabla.index[0]
print("mejor:", mejor)

## 10. Salida diaria: exportación y métricas de arbitraje

El MAE mide precisión de precio. Para el capítulo de baterías importa algo distinto: **si el
modelo acierta las horas de mínimo y máximo**, porque de ahí sale el margen. Un modelo con peor
MAE que clave las horas vale más para operar que uno preciso que las confunda.

La **captura de spread** mide qué fracción del arbitraje perfecto se obtendría cargando en la
hora de valle predicha y descargando en la de pico predicha. Es el número que conecta este
capítulo con el de optimización, y probablemente el más contundente de la defensa: un 85% de
captura dice más a un tribunal que cualquier MAE.

Genera tres ficheros: resumen por día, predicciones hora a hora, y la comparativa de modelos.

In [ ]:
M, NOMBRE_M = te, "test"          # durante la iteracion, cambiar a va / "validacion"
p, yt = preds[mejor], y[M]
fm = pd.to_datetime(fechas[M])

h_min_r, h_max_r = yt.argmin(1), yt.argmax(1)
h_min_p, h_max_p = p.argmin(1), p.argmax(1)
n = np.arange(len(yt))
spread_perfecto = yt.max(1) - yt.min(1)
spread_capturado = yt[n, h_max_p] - yt[n, h_min_p]     # spread REAL de las horas ELEGIDAS

diario = pd.DataFrame({
    "fecha": fm.date,
    "precio_real_medio": yt.mean(1).round(2),
    "precio_pred_medio": p.mean(1).round(2),
    "MAE_dia": np.abs(p - yt).mean(1).round(2),
    "sesgo_dia": (p - yt).mean(1).round(2),
    "hora_min_real": h_min_r, "hora_min_pred": h_min_p,
    "hora_max_real": h_max_r, "hora_max_pred": h_max_p,
    "spread_perfecto": spread_perfecto.round(2),
    "spread_capturado": spread_capturado.round(2),
    "captura_pct": (100 * spread_capturado / np.maximum(spread_perfecto, .1)).round(1),
})
diario.to_csv(f"resultados_diarios_{NOMBRE_M}.csv", index=False)

horario = pd.DataFrame({"fecha": np.repeat(fm.date, 24),
                        "hora": np.tile(np.arange(24), len(fm)),
                        "real": yt.ravel().round(2)})
for k in preds:
    horario[k] = preds[k].ravel().round(2)
horario["error_mejor"] = (horario[mejor] - horario["real"]).round(2)
horario.to_csv(f"predicciones_horarias_{NOMBRE_M}.csv", index=False)
tabla.round(3).to_csv("comparativa_modelos.csv")

print(f"modelo: {mejor}   conjunto: {NOMBRE_M}   {len(diario)} dias\n")
print(f"  MAE medio diario      {diario.MAE_dia.mean():.2f} EUR/MWh")
print(f"  peor dia (p95)        {np.percentile(diario.MAE_dia, 95):.2f}")
print(f"  hora de valle         {100*(h_min_r==h_min_p).mean():.0f}% exacta | "
      f"{100*(np.abs(h_min_r-h_min_p)<=1).mean():.0f}% +-1h")
print(f"  hora de pico          {100*(h_max_r==h_max_p).mean():.0f}% exacta | "
      f"{100*(np.abs(h_max_r-h_max_p)<=1).mean():.0f}% +-1h")
print(f"  captura de spread     {diario.captura_pct.mean():.1f}% del arbitraje perfecto")
print(f"\nficheros: resultados_diarios_{NOMBRE_M}.csv, predicciones_horarias_{NOMBRE_M}.csv, "
      f"comparativa_modelos.csv")
display(diario.head(10))

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(15, 8))

ax[0,0].plot(fm, yt.mean(1), color="black", lw=1, label="real")
ax[0,0].plot(fm, p.mean(1), lw=1, alpha=.85, label=mejor)
ax[0,0].set_title("Precio medio diario"); ax[0,0].set_ylabel("EUR/MWh"); ax[0,0].legend()

ax[0,1].plot(fm, diario.MAE_dia, lw=.9, color="firebrick")
ax[0,1].axhline(diario.MAE_dia.mean(), ls="--", color="grey")
ax[0,1].set_title(f"MAE por día (media {diario.MAE_dia.mean():.1f} EUR/MWh)")

ax[1,0].hist(h_max_p - h_max_r, bins=np.arange(-12, 13) - .5, color="steelblue")
ax[1,0].set_title("Error en la hora de pico (0 = acierto)"); ax[1,0].set_xlabel("horas")

ax[1,1].hist(diario.captura_pct, bins=30, color="seagreen")
ax[1,1].axvline(diario.captura_pct.mean(), color="r", ls="--")
ax[1,1].set_title(f"Captura de spread (media {diario.captura_pct.mean():.0f}%)")
ax[1,1].set_xlabel("% del arbitraje perfecto")

plt.tight_layout(); plt.show()

## 11. Lectura de resultados y siguientes pasos

**La referencia.** El naive D (repetir el precio de hoy) da **17,13 €/MWh de MAE** sobre test.
El sMAPE del 72% no indica un mal baseline: es alto porque el precio pasa cerca de cero en horas
de excedente solar y ahí cualquier error relativo se dispara. El MAE manda sobre el sMAPE.

**El hallazgo principal.** Predecir el **residuo frente al naive** en lugar del precio absoluto
es lo que separa "no bate al baseline" de "lo bate". En escala absoluta ningún modelo llegaba
(MLP 20,9 · LSTM 23,9 · Seq2Seq 27,2 frente a 17,1); con residuo, el Seq2Seq baja a 15,1. La
causa está diagnosticada: train incluye la crisis del gas de 2022 con una media de 93 €/MWh, y
la comprobación de escalado muestra que el nivel de test está medio sigma por debajo. Una red
regularizada tira hacia la media del target que ha visto; el naive es inmune porque no aprende
ningún nivel. **Al cambiar el target a la desviación, el problema de régimen desaparece.** Esto
es material de memoria, no un detalle de implementación.

**Método — el test se abre una vez.** Se itera contra validación (2025) y se reserva test (2026)
para el número final. Cada vez que se mira el test y se decide en función de lo que se ve, se
contamina; tras veinte iteraciones eligiendo "lo que va mejor en test", ese MAE ya no estima
nada. Las secciones 8b y 10 tienen una variable `MASCARA` / `M` justamente para trabajar sobre
validación mientras se itera.

**Si LightGBM gana al Seq2Seq, se reporta.** Con ~1.800 días y datos tabulares los árboles son
muy difíciles de batir. Decirlo con honestidad vale más que forzar el resultado contrario; el
deep learning tendrá su justificación cuando entre el tensor meteorológico, donde un árbol no
puede competir.

**Limitaciones a documentar.**

- Batería: ESIOS 2166/2167 no publican hasta el 20-nov-2024 y train acaba el 31-dic-2024, así
  que son constantes en casi todo el entrenamiento — de hecho la §3 las descarta por falta de
  varianza. No se puede modelar el efecto de la batería sobre el precio con estos datos, lo que
  refuerza el planteamiento del capítulo de optimización, donde es variable de decisión y no
  feature explicativa.
- Excepción ibérica (jun-2022 a dic-2023): entera dentro de train y fuera de validación y test.
  Desplazamiento de régimen por construcción, mitigado con el dummy, el residuo y el
  fine-tuning.
- Desde oct-2025 el precio horario es la media de cuatro MTU de 15 minutos distintos; la
  varianza intrahoraria se descarta.
- Ocho configuraciones de hiperparámetros y no más: con 357 días de validación, una búsqueda
  amplia sobreajustaría la propia validación.

**Siguiente iteración, por orden de retorno esperado.**

1. **Encoder con generación real hora a hora.** Ahora `X_enc` es solo precio, forma (168, 1). El
   `construir_tensores()` del script genera `X_hist` con 22 canales: generación por tecnología,
   demanda, flujos. Es el salto grande que queda y vale más que cualquier ajuste de dropout.
2. Ablaciones — sin previsiones, sin commodities, sin PDBC, con ERA5 perfecto. Cada una es un
   párrafo de la memoria con un número detrás. La de ERA5 cuantifica el valor de mejorar la
   previsión meteorológica y justifica el punto 4.
3. Atención sobre el decoder (patrón `BahdanauAttention` de `img2seq`): da los mapas del
   capítulo de interpretabilidad.
4. Tensor meteorológico ECMWF como entrada convolucional — el elemento diferencial del TFM y el
   único punto donde la GPU de Colab compensa de verdad.

## 12. Comparar matrices y ablación del encoder

Las dos tablas que van al capítulo. Ejecuta el notebook entero cambiando las constantes de la
celda de configuración; cada ejecución deja su `registro_*.json` y la última celda las junta.

| Ejecución | `MATRIZ` | `ENCODER_MULTICANAL` | Qué mide |
|---|---|---|---|
| A | `nucleo` | `True` | referencia |
| B | `completa` | `True` | ¿aportan las 29 columnas que el núcleo poda? |
| C | `minima` | `True` | ¿bastan las 25 mejores? |
| D | `moderna` | `True` | **¿estorba la crisis del gas?** |
| E | `nucleo` | `False` | cuánto aporta el encoder multicanal |

**Compara en validación, no en test.** Aquí estás eligiendo entre alternativas, y cada vez que
se mira el test para decidir algo, se contamina.

**D es la que tiene más que contar.** `moderna` lleva exactamente el mismo pool que `nucleo` y
sólo cambia la ventana: empieza en 2023 en lugar de 2020. Así que la diferencia entre A y D
mide una cosa y sólo una — si los 26.000 registros de 2020-2022 ayudan o estorban. Y hay razón
para dudar: 2022 promedió 167,5 €/MWh, **2,66 veces** los 63,0 de 2024, con el tope al gas
activo el 55 % del año. Si D gana con 17.566 días de train frente a los 43.699 de A, el TFM
tiene un resultado propio que defender.

Al leer A contra B, ten presente que el núcleo poda por redundancia (27 columnas) y por deriva
train→validación (12). Si B gana, alguno de esos dos criterios está cortando de más, y la
tabla de la sección 10 dice cuál.

In [ ]:
# Guarda el resultado de esta ejecucion para poder compararlo con las demas
import json as _json

reg = {"matriz": MATRIZ, "hash_matriz": META.get("hash", "?"),
       "generada": META.get("generada", "?"),
       "encoder_multicanal": ENCODER_MULTICANAL,
       "aisla": META["aisla"],
       "canales": len(canales), "estaticos": int(Xs.shape[1]),
       "n_inputs": META["n_inputs"],
       "dias_train": int(tr.sum()), "dias_val": int(va.sum())}

# MAE en VALIDACION de cada modelo -- es lo correcto para elegir entre alternativas; el test
# se reserva para el numero final de la memoria
mae_val = {}
for nombre, modelo, entrada in [("MLP residuo", mlp_r, "plano"),
                                ("Seq2Seq residuo", s2r, "ent")]:
    try:
        p = (modelo.predict(plano[va], verbose=0) if entrada == "plano"
             else modelo.predict(ent(va), verbose=0))
        mae_val[nombre] = round(float(metricas(y[va], inv_r(p, va))["MAE"]), 2)
    except Exception as e:
        print(f"  ({nombre} no disponible: {e})")
mae_val["naive D"] = round(float(metricas(y[va], naive_todos[va])["MAE"]), 2)
reg["MAE_val"] = mae_val

etiqueta = f"{MATRIZ}_{'multi' if ENCODER_MULTICANAL else 'solo_precio'}"
Path(f"registro_{etiqueta}.json").write_text(_json.dumps(reg, indent=2), encoding="utf-8")
print(_json.dumps(reg, indent=2))

# tabla acumulada con todas las ejecuciones guardadas
regs = [_json.loads(p.read_text(encoding="utf-8"))
        for p in sorted(Path(".").glob("registro_*.json"))]
if len(regs) > 1:
    filas = [{"matriz": r.get("matriz", r.get("variante", "?")),
              "hash": r.get("hash_matriz", "?"),
              "encoder": r["canales"], "inputs": r.get("n_inputs"),
              "estaticos": r["estaticos"], "dias_train": r["dias_train"],
              **r["MAE_val"]} for r in regs]
    t = pd.DataFrame(filas)
    display(t)
    # OJO al leer esta tabla: `moderna` entrena con menos dias que las demas. Un MAE
    # ligeramente peor con la mitad de datos no es un empate, es una señal a favor.
    if {"nucleo", "moderna"} <= set(t["matriz"]):
        print()
        print("nucleo vs moderna: misma columnas, distinta ventana -- la comparacion limpia")
        print(t[t.matriz.isin(["nucleo", "moderna"])].to_string(index=False))